# RPO

- Arbitrary RAAN, inclination, and phasing
- Angles, then 3 DOF, then 6 DOF

The original step-by-step plan lives in `rpo_legacy.ipynb`.

In [55]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path

import numpy as np
from cesium.constants import (
    HOUR_TO_SEC,
    SEC_TO_MIN,
    DEG_TO_RAD,
    KM_TO_M,
    M_TO_KM,
    MIN_TO_SEC,
    RAD_TO_DEG,
)
from cesium.guidance.hcw import RpoConfig, dv_breakdown, dv_breakdown_table, run_rpo
from cesium.math.time import jd_to_greg, jd_to_sec_j2000
from cesium.orbital.coes import COES
from cesium.plotting import plot_3d, plot_rsw, plot_separation
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from numpy.linalg import norm

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [57]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_0

dt = 1  # sec

In [58]:
# Chaser = target + seeded uniform dispersion
# SEED = 525600
# SEED = 3214359876
SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
J2 = 0
INITIAL_TA_OFFSET_DEG = 57

# CRAZY
# ANGLE_DISP_DEG = 50.0  # i, raan, aop
# A_DISP_M = 500 * KM_TO_M
J2 = J2_MARS

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_0 = COES.from_array(TARGET_0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_0

TARGET_0.print("initial target coes")
CHASER_0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3768.960 km, e = 0.00855, i = 46.866 deg, raan = 13.569 deg, aop = -2.537 deg, ta = 70.000 deg


In [59]:
# From arrival.ipynb
MISSION_NAME = "birthday-christmas-earlier"
mission = np.load(f"trajectories/{MISSION_NAME}.mission", allow_pickle=False)
EPOCH0_JD_TDB = float(mission["lambert_arrival_jd"])
EPOCH0_TDB_S = jd_to_sec_j2000(EPOCH0_JD_TDB)

print(
    "RPO start: {:04.0f}-{:02.0f}-{:02.0f} {:02.0f}:{:02.0f}:{:06.3f} TDB".format(
        *jd_to_greg(EPOCH0_JD_TDB)
    ),
    f"({EPOCH0_TDB_S:.3f} s past J2000)",
)

RPO start: 2035-12-25 00:00:00.000 TDB (1135425600.000 s past J2000)


## Burn timeline
One propagated truth for both spacecraft (`cesium.guidance.RpoTimeline`): every `coast` continues from the last state, every `burn` is applied to it, and each burn is planned from the current (propagated) COES. Times are seconds since `EPOCH0_TDB_S`; `tl.epochs_tdb_s()` and `tl.burn_table()` give absolute epochs.

- LOWER ORBITS DO RAAN FASTER

In [60]:
def raan_dot_j2(a, e, i):
    """Secular J2 nodal regression rate [rad/s]."""
    n = np.sqrt(MU_MARS / a**3)
    p = a * (1 - e**2)
    return -1.5 * n * J2 * (R_MARS / p) ** 2 * np.cos(i)

In [ ]:
CFG = RpoConfig(
    mu=MU_MARS,
    j2=J2,
    r_cb=R_MARS,
    epoch0_tdb_s=EPOCH0_TDB_S,
    k_phase=20,
    hcw_thresh=20 * KM_TO_M,
    t_rendezvous=40 * MIN_TO_SEC,
    dock_thresh=50,  # m
)

# Pass 1: no bias, to measure the J2 differential nodal drift accumulated during phasing
tl_0, info_0 = run_rpo(CHASER_0, TARGET_0, CFG, verbose=True)

# Analytic estimate: phasing orbit sits higher, so its node regresses more slowly
drift_analytic = (
    raan_dot_j2(info_0["a_phase"], 0, info_0["i_target"])
    - raan_dot_j2(info_0["a_target"], 0, info_0["i_target"])
) * info_0["t_phase"]
print(
    f"RAAN drift during phasing: measured {info_0['draan'] * RAD_TO_DEG:.4f} deg, "
    f"analytic {drift_analytic * RAD_TO_DEG:.4f} deg\n"
)

# Pass 2: pre-bias Burn 1 by the measured drift
raan_bias = -info_0["draan"]
print(f"Re-running with RAAN bias {raan_bias * RAD_TO_DEG:.4f} deg")
tl, info = run_rpo(CHASER_0, TARGET_0, CFG, raan_bias=raan_bias, verbose=True)
n_hat = info["n_hat"]

print(f"\nJ2 bias saves {tl_0.dv_total - tl.dv_total:.3f} m/s")

ts_tl, rv_c_tl, rv_t_tl = tl.arrays()

# Separation vs time: shaded phases labeled with the burns they span, plus a tick per burn
def short(name):
    """'Burn 4.3 (phasing)' -> 'Burn 4.3'."""
    return name.split(" (")[0]


rows = {r["phase"]: r for r in dv_breakdown(tl)}
bands = [
    ("plane change + Hohmann", ["Plane change", "Hohmann + phase entry"], "#e8eef9", "top left"),
    ("phasing", ["Phasing"], "#f1f1ec", "top"),
    ("phase exit + HCW", ["Phase exit", "HCW terminal"], "#fbe9e7", "top right"),
]
t_dock = rows["HCW terminal"]["t_end"]
fig_sep = plot_separation(
    ts_tl, rv_c_tl, rv_t_tl, left=0, right=(t_dock + 2 * HOUR_TO_SEC) * SEC_TO_MIN, show=False
)
for label, phases, fill, anchor in bands:
    rs = [rows[ph] for ph in phases if ph in rows]
    t0, t1 = rs[0]["t_start"], rs[-1]["t_end"]
    span = f"{short(rs[0]['first_burn'])} – {short(rs[-1]['last_burn'])}"
    fig_sep.add_vrect(
        x0=t0 * SEC_TO_MIN,
        x1=max(t1, t0 + 20 * 60) * SEC_TO_MIN,  # min width so short phases show
        fillcolor=fill,
        opacity=0.9,
        layer="below",
        line_width=0,
        annotation_text=f"<b>{span}</b><br>{label}<br>{sum(r['dv'] for r in rs):.1f} m/s",
        annotation_position=anchor,
        annotation_font={"size": 12, "color": "#52514e"},
    )

# Tick per burn along the time axis (hover for name and dv)
fig_sep.add_scatter(
    x=[b.t_s * SEC_TO_MIN for b in tl.burns],
    y=[0.0] * len(tl.burns),
    mode="markers",
    marker={"symbol": "line-ns", "size": 14, "line": {"width": 2, "color": "#52514e"}},
    text=[f"{b.name}<br>{b.dv_mag:.2f} m/s" for b in tl.burns],
    hovertemplate="%{text}<br>t = %{x:.1f} min<extra></extra>",
    cliponaxis=False,
)

# Labeled callouts for the non-phasing burns, fanned out so close burns stay readable
key = [b for b in tl.burns if "(phasing)" not in b.name]
t_mid = 0.5 * t_dock
left = [b for b in key if b.t_s < t_mid]
right = [b for b in key if b.t_s >= t_mid]
for group, sign in ((left, 1), (right, -1)):
    for k, b in enumerate(group):
        fig_sep.add_annotation(
            x=b.t_s * SEC_TO_MIN,
            y=0,
            text=f"{b.name}: {b.dv_mag:.1f} m/s",
            showarrow=True,
            arrowhead=0,
            arrowcolor="#898781",
            ax=sign * (30 + 15 * k),
            ay=-35 - 30 * k,
            xanchor="left" if sign > 0 else "right",
            font={"size": 11, "color": "#52514e"},
        )
fig_sep.update_layout(title=f"Chaser-to-target separation (total Δv {tl.dv_total:.1f} m/s)")
fig_sep.show()

### HCW Targeting

In [ ]:
print(dv_breakdown_table(tl))
print()
print(tl.burn_table())
print()
for b in tl.burns:
    if b.name.startswith("HCW"):
        rr = tl.rel_at(b.t_s)
        print(f"at {b.name:<16}: |r| = {norm(rr[0:3]):9.2f} m, |v| = {norm(rr[3:6]):.4f} m/s")

In [ ]:
# Reference circles around the target
RSW_CIRCLES = [
    {"radius": 15 * KM_TO_M, "name": "HCW linear region (15 km)", "color": "#7a7a74", "dash": "dot"},
    {"radius": 200, "name": "Keep-out (200 m)", "color": "#d03b3b", "dash": "dash"},
]

ts_rsw, rv_rsw = tl.rsw_arrays(progress=True)
burns_tl = tl.burn_events()

# Full RPO, with Mars's surface for altitude context
fig_rsw = plot_rsw(
    ts_rsw[::5],
    rv_rsw[::5],
    burns_tl,
    title="RPO in the target's RSW frame",
    rho=TARGET_0.a,
    r_body=R_MARS,
    circles=RSW_CIRCLES,
)

# Terminal approach: phase exit through the HCW stop
t_exit = next(b.t_s for b in tl.burns if b.name.startswith("Burn 5"))
t_stop = next(b.t_s for b in tl.burns if b.name == "HCW stop")
m = (ts_rsw >= t_exit) & (ts_rsw <= t_stop)
fig_rsw_terminal = plot_rsw(
    ts_rsw[m],
    rv_rsw[m],
    [(t_b, name) for t_b, name in burns_tl if t_exit <= t_b <= t_stop],
    title="Terminal approach: phase exit and HCW rendezvous",
    circles=RSW_CIRCLES,
)

# Close-up: last 1 km, against the keep-out
m_close = m & (norm(rv_rsw[:, 0:3], axis=1) < 1 * KM_TO_M)
fig_rsw_close = plot_rsw(
    ts_rsw[m_close],
    rv_rsw[m_close],
    [(t_b, name) for t_b, name in burns_tl if t_b >= ts_rsw[m_close][0] and t_b <= t_stop],
    title="Final approach (inside 1 km)",
    circles=RSW_CIRCLES[1:],
)

In [ ]:
# 3D view of the chaser in the target's RSW frame (plot X/Y/Z = R/S/W), target at the origin
# Zoom on HCW: m = ts_rsw >= next(t_b for t_b, name in burns_tl if name.startswith("HCW"))
m = np.ones_like(ts_rsw, dtype=bool)
r_rel_rsw = rv_rsw[m, 0:3]
ts_rel = ts_rsw[m]

axis_len = 0.01 * norm(r_rel_rsw, axis=1).max()  # [m]
rsw_axes = [
    {
        "direction": [1, 0, 0],
        "length": axis_len,
        "name": "R",
        "color": "red",
        "label": True,
    },
    {
        "direction": [0, 1, 0],
        "length": axis_len,
        "name": "S",
        "color": "green",
        "label": True,
    },
    {
        "direction": [0, 0, 1],
        "length": axis_len,
        "name": "W",
        "color": "blue",
        "label": True,
    },
]

burn_points = [
    {"name": "Target", "location": np.zeros(3), "color": "black", "symbol": "x"}
]
for t_b, name in burns_tl:
    if t_b >= ts_rel[0] and "(phasing)" not in name:  # 20 phasing burns would flood the legend
        k = np.argmin(np.abs(ts_rel - t_b))
        burn_points.append(
            {"name": name, "location": r_rel_rsw[k], "color": "orange", "size": 4, "label": False}
        )
fig_rsw_3d = plot_3d(
    r_rel_rsw[::5],
    ts_rel[::5],
    labels="Chaser (rel. target)",
    trace_colors="royalblue",
    title="Chaser relative to target (RSW: X = R, Y = S, Z = W)",
    vectors=rsw_axes,
    points=burn_points,
    scale=KM_TO_M,  # inputs in m, displayed in km
    units="km",
)

## Visualization

In [ ]:
vectors = [
    {
        "direction": n_hat,
        "length": 1.6 * R_MARS * M_TO_KM,  # km, just past the surface
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

fig_mci = plot_3d(
    [rv_t_tl[:, 0:3] * M_TO_KM, rv_c_tl[:, 0:3] * M_TO_KM],
    [ts_tl, ts_tl],
    labels=["Target", "Chaser"],
    trace_colors=["black", "royalblue"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title="Target and chaser, full timeline (MCI)",
    vectors=vectors,
    units="km",
)

## Export
Static figures and the Δv breakdown for slides.

In [ ]:
FIG_DIR = Path("figures/rpo")
FIG_DIR.mkdir(parents=True, exist_ok=True)

slide_figs = {
    "separation": fig_sep,
    "rsw_full": fig_rsw,
    "rsw_terminal": fig_rsw_terminal,
    "rsw_close": fig_rsw_close,
    "rsw_3d": fig_rsw_3d,
    "mci_3d": fig_mci,
}
for name, f in slide_figs.items():
    f.write_image(FIG_DIR / f"{name}.png", width=1600, height=f.layout.height or 800, scale=2)
    print(f"wrote {FIG_DIR / name}.png")

(FIG_DIR / "dv_breakdown.txt").write_text(dv_breakdown_table(tl) + "\n")
(FIG_DIR / "burn_table.txt").write_text(tl.burn_table() + "\n")
print(f"wrote {FIG_DIR}/dv_breakdown.txt, burn_table.txt")

# end